# 12 并行策略选择指南

## 概述

大语言模型的训练和推理需要大量计算资源，单张 GPU 往往无法满足需求。
**选择合适的并行策略**是高效利用集群资源的关键——错误的策略会导致显存不足或通信开销过大。

本笔记本将系统对比六大并行策略，提供决策树和推荐配置，帮助你在不同场景下做出最优选择。

### 🧠 直觉理解：选择并行策略就像选择出行方式

不同的出行距离需要不同的交通工具：

- **短途走路（数据并行 DP）**：距离近、不需要交通工具——模型能放进单卡，直接多卡复制
- **中途开车（张量并行 TP）**：距离适中、需要一辆车——模型稍大，切分到同机多卡
- **长途坐飞机（PP+TP+DP）**：距离远、需要多种交通方式——超大模型，组合多种并行

选择错误的出行方式（比如短途坐飞机）会浪费资源；选择正确的并行策略能最大化效率。

### 📐 数学原理：通信开销 vs 显存节省的权衡

并行策略的核心权衡是**通信开销**与**显存节省**之间的平衡。

**通信开销**：设每个 GPU 的计算量为 $C$，通信量为 $M$，则并行效率为：

$$\eta = \frac{C / N}{C / N + M \cdot \alpha + M \cdot \beta}$$

其中 $N$ 是 GPU 数量，$\alpha$ 是延迟（latency），$\beta$ 是带宽倒数（$1/\text{bandwidth}$）。

**显存节省**：设单卡显存需求为 $M_{\text{single}}$，使用 $N$ 路并行后的显存为：

$$M_{\text{per\_gpu}} = \frac{M_{\text{single}}}{S} + M_{\text{comm}}$$

其中 $S$ 是分片因子，$M_{\text{comm}}$ 是通信缓冲区开销。

**关键洞察**：节点内通信（NVLink, 900 GB/s）比节点间通信（InfiniBand, ~50 GB/s）快约 18 倍，
因此 TP 适合节点内，PP 适合节点间。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("并行策略选择指南 - 可视化分析")

## 1. 六大并行策略对比

| 策略 | 全称 | 核心思想 | 显存节省 | 通信开销 | 适用场景 |
|------|------|----------|----------|----------|----------|
| DP | 数据并行 | 数据分片，模型复制 | 低 | 低 | 模型可放入单卡 |
| TP | 张量并行 | 权重列切分 | 高 | 高 | 节点内多卡 |
| PP | 流水线并行 | 层间切分 | 中 | 中 | 多节点训练 |
| EP | 专家并行 | MoE 专家分片 | 高 | 高 | MoE 模型 |
| CP | 上下文并行 | 序列维度切分 | 高 | 中 | 超长序列 |
| 推理优化 | - | KV Cache 分片等 | 中 | 低 | 推理服务 |

In [ ]:
strategies = ['DP', 'TP', 'PP', 'EP', 'CP', '推理优化']
memory_saving = [0, 0.75, 0.5, 0.75, 0.75, 0.5]
comm_overhead = [0.1, 0.5, 0.3, 0.6, 0.4, 0.2]

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(comm_overhead, memory_saving, s=200, c=range(len(strategies)), cmap='Set2', zorder=5)
for i, s in enumerate(strategies):
    ax.annotate(s, (comm_overhead[i], memory_saving[i]), textcoords="offset points", xytext=(10, 5), fontsize=12)
ax.set_xlabel('通信开销')
ax.set_ylabel('显存节省')
ax.set_title('六大并行策略：通信开销 vs 显存节省')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 并行策略选择决策树

根据模型大小、GPU 显存、集群规模选择并行策略：

```
模型能否放入单卡？
├── 是 → 数据并行 (DP)
│   └── 数据量是否足够大？
│       ├── 是 → 增加 DP 度
│       └── 否 → 增大 batch size / 梯度累积
└── 否 → 模型能否放入单节点？
    ├── 是 → 张量并行 (TP)
    │   └── TP 度 = 模型显存 / 单卡显存（向上取整）
    └── 否 → TP + 流水线并行 (PP)
        ├── TP 度 = 节点内 GPU 数
        └── PP 度 = 总 GPU 数 / TP 度
            └── 是否为 MoE 模型？
                ├── 是 → 加入专家并行 (EP)
                └── 否 → 是否超长序列？
                    ├── 是 → 加入上下文并行 (CP)
                    └── 否 → TP + PP + DP
```

In [ ]:
# 决策树可视化
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('并行策略选择决策树', fontsize=14, fontweight='bold')

# 决策节点
def draw_box(ax, x, y, text, color='#3498db', w=2.2, h=0.6):
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, facecolor=color, alpha=0.2, edgecolor=color, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.15, my, label, fontsize=8, color='#e74c3c', fontweight='bold')

# 顶层
draw_box(ax, 6, 6.3, '模型能放入单卡？', '#e74c3c')
draw_arrow(ax, 4.5, 6, 3, 5.3, '是')
draw_arrow(ax, 7.5, 6, 9, 5.3, '否')

# 第二层
draw_box(ax, 3, 5, 'DP', '#2ecc71')
draw_box(ax, 9, 5, '能放入单节点？', '#e74c3c')
draw_arrow(ax, 7.5, 4.7, 6, 4, '是')
draw_arrow(ax, 10.5, 4.7, 11, 4, '否')

# 第三层
draw_box(ax, 6, 3.7, 'TP', '#3498db')
draw_box(ax, 11, 3.7, 'TP + PP', '#9b59b6')
draw_arrow(ax, 11, 3.4, 11, 2.7)

# 第四层
draw_box(ax, 11, 2.4, 'MoE?', '#e74c3c')
draw_arrow(ax, 9.5, 2.1, 8.5, 1.5, '是')
draw_arrow(ax, 12.5, 2.1, 11, 1.5, '否')

# 第五层
draw_box(ax, 8.5, 1.2, 'TP+PP+EP', '#f39c12')
draw_box(ax, 11, 1.2, '超长序列?', '#e74c3c')
draw_arrow(ax, 9.5, 0.9, 8.5, 0.3, '否')
draw_arrow(ax, 12.5, 0.9, 11, 0.3, '是')

# 第六层
draw_box(ax, 8.5, 0, 'TP+PP+DP', '#2ecc71')
draw_box(ax, 11, 0, 'TP+PP+CP+DP', '#1abc9c')

plt.tight_layout()
plt.show()

## 3. 不同规模下的推荐组合

不同参数规模的模型需要不同的并行配置。以下以 A100-80G 为基准：

| 模型规模 | 参数量 | 单卡显存需求 | 推荐配置 | GPU 数量 |
|----------|--------|-------------|----------|----------|
| 7B | 70亿 | ~28 GB | DP | 1-8 |
| 13B | 130亿 | ~52 GB | TP=2 + DP | 2-8 |
| 70B | 700亿 | ~280 GB | TP=8 + PP=2 + DP | 16+ |
| 405B | 4050亿 | ~1.6 TB | TP=8 + PP=4 + DP | 64+ |

In [ ]:
models = ['7B\n(单机)', '13B\n(单机多卡)', '70B\n(多机)']
configs = ['DP', 'TP+DP', 'TP+PP+DP']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(models, [1, 2, 3], color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_ylabel('并行策略复杂度')
ax.set_title('不同规模模型的推荐并行配置')
for bar, config in zip(bars, configs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05, config, ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 4. 成本-收益分析

并行策略的选择本质上是**通信开销**与**显存节省**之间的权衡。

下面对比不同策略在 A100 集群上的理论效率：
- **节点内**：NVLink 带宽 900 GB/s，TP 通信开销小
- **节点间**：InfiniBand 带宽 ~50 GB/s，PP 通信开销可控
- **DP**：仅需 AllReduce 梯度，通信量与模型大小成正比

In [ ]:
# 通信开销 vs 显存节省的详细对比
strategies_detail = {
    'DP': {'mem_save': 0.0, 'comm': 0.1, 'scale': '数据量', 'bandwidth': '高'},
    'TP': {'mem_save': 0.75, 'comm': 0.5, 'scale': '模型大小', 'bandwidth': '极高(NVLink)'},
    'PP': {'mem_save': 0.5, 'comm': 0.3, 'scale': '激活值', 'bandwidth': '中(IB)'},
    'EP': {'mem_save': 0.75, 'comm': 0.6, 'scale': '专家数', 'bandwidth': '高(All-to-All)'},
    'CP': {'mem_save': 0.75, 'comm': 0.4, 'scale': '序列长度', 'bandwidth': '中'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：雷达图替代 - 分组柱状图
names = list(strategies_detail.keys())
mem_saves = [strategies_detail[k]['mem_save'] for k in names]
comm_costs = [strategies_detail[k]['comm'] for k in names]

x = np.arange(len(names))
width = 0.35
axes[0].bar(x - width/2, mem_saves, width, label='显存节省', color='#2ecc71')
axes[0].bar(x + width/2, comm_costs, width, label='通信开销', color='#e74c3c')
axes[0].set_xlabel('并行策略')
axes[0].set_ylabel('归一化值')
axes[0].set_title('显存节省 vs 通信开销')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：效率 = 显存节省 / (1 + 通信开销)
efficiency = [m / (1 + c) for m, c in zip(mem_saves, comm_costs)]
colors = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12', '#1abc9c']
axes[1].bar(names, efficiency, color=colors)
axes[1].set_xlabel('并行策略')
axes[1].set_ylabel('综合效率')
axes[1].set_title('并行策略综合效率 (显存节省 / (1+通信开销))')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 实际案例分析：LLaMA 3 并行配置

### LLaMA 3 8B
- **参数量**：80 亿
- **显存需求**：~28 GB（fp16 训练 + Adam 优化器）
- **推荐配置**：单机 8×A100-80G，纯 DP
- **吞吐量**：~4000 tokens/s/GPU

### LLaMA 3 70B
- **参数量**：700 亿
- **显存需求**：~280 GB
- **推荐配置**：2 节点 16×A100-80G，TP=8 + PP=2 + DP=1
- **关键**：TP 在节点内（NVLink），PP 在节点间（IB）

### LLaMA 3 405B
- **参数量**：4050 亿
- **显存需求**：~1.6 TB
- **推荐配置**：8 节点 64×A100-80G，TP=8 + PP=4 + DP=2
- **额外优化**：ZeRO-3 分片优化器状态、激活重计算

> **经验法则**：优先使用 TP 消耗节点内 GPU，再用 PP 跨节点，最后用 DP 扩大规模。

In [ ]:
# LLaMA 3 不同规模的并行配置可视化
fig, ax = plt.subplots(figsize=(10, 5))

model_sizes = ['8B', '70B', '405B']
gpu_counts = [8, 16, 64]
tp_degrees = [1, 8, 8]
pp_degrees = [1, 2, 4]
dp_degrees = [8, 1, 2]

x = np.arange(len(model_sizes))
width = 0.25

bars1 = ax.bar(x - width, tp_degrees, width, label='TP 度', color='#3498db')
bars2 = ax.bar(x, pp_degrees, width, label='PP 度', color='#9b59b6')
bars3 = ax.bar(x + width, dp_degrees, width, label='DP 度', color='#2ecc71')

ax.set_xlabel('模型规模')
ax.set_ylabel('并行度')
ax.set_title('LLaMA 3 不同规模的并行配置')
ax.set_xticks(x)
ax.set_xticklabels(model_sizes)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 标注总 GPU 数
for i, (g, t, p, d) in enumerate(zip(gpu_counts, tp_degrees, pp_degrees, dp_degrees)):
    ax.text(i, max(t, p, d) + 0.5, f'共 {g} GPU', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 📝 练习题

1. **策略选择**：假设你要训练一个 30B 参数的模型，有 4 台 8×A100-80G 服务器。你会选择什么并行配置？请说明理由，并计算 TP、PP、DP 的度数。

2. **通信瓶颈分析**：在 TP=8 的配置中，每次前向传播需要多少次 AllReduce？如果换成 PP=8，通信模式有何不同？哪种方式在节点间更高效？

3. **MoE 模型并行**：DeepSeek-V3 是一个 671B 参数的 MoE 模型（每次激活 37B）。为什么 EP 对 MoE 模型特别重要？如果不使用 EP，MoE 模型的显存瓶颈在哪里？